# Products Data Ingestion & Transformation Pipeline

## Overview
This notebook presents the data pipeline for processing product data across the **Medallion Architecture layers (Bronze -> Silver -> Gold)** using Databricks, PySpark, and Delta Lake.

---

In [0]:
### 1. Setup & Environment Configurations

from pyspark.sql import functions as F
from delta.tables import DeltaTable

In [0]:
%run "../Utitlites"

In [0]:
# ==========================================
# 2. Dynamic Notebook Parameters (Widgets)
# ==========================================
dbutils.widgets.text('catalog', 'fmcg', 'Catalog')
dbutils.widgets.text('data_Sourse', 'customers', 'Data Source')

# Extract parameter values
catalog = dbutils.widgets.get('catalog')
data_source = dbutils.widgets.get('data_Sourse')

 
## Data Ingestion
Read raw CSV files, append metadata attributes, and add timestamps.

In [0]:
# Read raw CSV file and attach metadata attributes
sb_products = (spark.read.format('csv')
        .option('header', True)
        .option('inferSchema', True)
        .load(f'{path_file}{data_source}.csv')
        .withColumn('read_timestamp', F.current_timestamp())
        .select('*', '_metadata.file_name', '_metadata.file_size')
)

# Display Schema
sb_products.printSchema()

root
 |-- product_name: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- category: string (nullable = true)
 |-- read_timestamp: timestamp (nullable = false)
 |-- file_name: string (nullable = false)
 |-- file_size: long (nullable = false)



## Bronze Layer:

In [0]:
# Save raw data to Bronze Delta Table
sb_products.write\
    .format('delta')\
    .mode('overwrite')\
    .option('delta.enableChangeDataFeed', 'true')\
    .saveAsTable(f'{catalog}.bronze.sb_dim_{data_source}')

# Show the Bronze Table
df_bronze = spark.sql(f'SELECT * FROM {catalog}.bronze.sb_dim_{data_source}')
df_bronze.show(10)

+--------------------+----------+-----------------+--------------------+------------+---------+
|        product_name|product_id|         category|      read_timestamp|   file_name|file_size|
+--------------------+----------+-----------------+--------------------+------------+---------+
|SportsBar Energy ...|  25891101|      energy bars|2026-09-28 06:51:...|products.csv|     1388|
|SportsBar Energy ...|  25891102|      energy bars|2026-09-28 06:51:...|products.csv|     1388|
|SportsBar Energy ...|  25891103|      energy bars|2026-09-28 06:51:...|products.csv|     1388|
|SportsBar Protien...|  25891201|     protien bars|2026-09-28 06:51:...|products.csv|     1388|
|SportsBar Protien...|  25891202|     protien bars|2026-09-28 06:51:...|products.csv|     1388|
|SportsBar Protien...|  25891203|     protien bars|2026-09-28 06:51:...|products.csv|     1388|
|SportsBar Granola...|  25891301|granola & cereals|2026-09-28 06:51:...|products.csv|     1388|
|SportsBar Granola...|  25891302|granola

In [0]:
# Drop Duplicates
# ------------------------------

df_bronze = df_bronze.dropDuplicates(['product_id'])

df_bronze.groupBy("product_id") \
    .agg(F.count('*').alias('count')) \
    .filter(F.col('count') > 1) \
    .show()

+----------+-----+
|product_id|count|
+----------+-----+
+----------+-----+



In [0]:
# Capitalize category values
df_bronze = df_bronze.withColumn(
    'category',
    F.when(F.col("category").isNull(), None).otherwise(F.initcap("category"))
)

# Mapping dictionary for typo correction
corrections_dict = {
    # 1. Protein Bars 
    "Protien Bars": "Protein Bars",
    "Proten Bars": "Protein Bars",
    "Portein Bars": "Protein Bars",
    "Protein Bar": "Protein Bars",
    "Proteinbars": "Protein Bars",
    
    # 2. Energy Bars
    "Enregy Bars": "Energy Bars",
    "Engery Bars": "Energy Bars",
    "Energi Bars": "Energy Bars",
    "Energy Bar": "Energy Bars",
    "Energybars": "Energy Bars",
    
    # 3. Granola & Cereals
    "Granola & Cereal": "Granola & Cereals",
    "Granola And Cereals": "Granola & Cereals",
    "Granola & Careals": "Granola & Cereals",
    "Granola Cerels": "Granola & Cereals",
    
    # 4. Recovery Dairy
    "Recovery Diary": "Recovery Dairy",
    "Recovery Dariy": "Recovery Dairy",
    "Recovry Dairy": "Recovery Dairy",
    "Recover Dairy": "Recovery Dairy",
    
    # 5. Healthy Snacks
    "Helthy Snacks": "Healthy Snacks",
    "Healthy Snaks": "Healthy Snacks",
    "Healthy Snak": "Healthy Snacks",
    "Health Snacks": "Healthy Snacks",
    
    # 6. Electrolyte Mix
    "Electrolite Mix": "Electrolyte Mix",
    "Electrolyte Mixes": "Electrolyte Mix",
    "Eletrolyte Mix": "Electrolyte Mix"
}

# Convert dictionary to DataFrame and join for mapping
category_correc = [(error, correct) for error, correct in corrections_dict.items()]
category_correc_table = spark.createDataFrame(category_correc, ['error', 'correct'])

df_bronze = df_bronze.join(
    F.broadcast(category_correc_table),
    df_bronze['category'] == category_correc_table['error'],
    'left'
).withColumn('category', F.coalesce(F.col('correct'), F.col('category'))\
).drop('error', 'correct')

# Show distinct categories
df_bronze.select("category").distinct().show()

+-----------------+
|         category|
+-----------------+
|   Healthy Snacks|
|      Energy Bars|
|Granola & Cereals|
|   Recovery Dairy|
|     Protein Bars|
|  Electrolyte Mix|
+-----------------+



In [0]:
# Division mapping logic
division_mapping = {
    "Energy Bars": "Nutrition Bars",
    "Protein Bars": "Nutrition Bars",
    "Granola & Cereals": "Breakfast Foods",
    "Recovery Dairy": "Dairy & Recovery",
    "Healthy Snacks": "Healthy Snacks",
    "Electrolyte Mix": "Hydration & Electrolytes"
}

mapping_list = [(key, val) for key, val in division_mapping.items()]
division_mapping_table = spark.createDataFrame(mapping_list, ['category_key', 'division_val'])

df_bronze = df_bronze.join(
    F.broadcast(division_mapping_table),
    df_bronze['category'] == division_mapping_table['category_key'],
    'left'
).withColumn(
    "division", F.coalesce(F.col("division_val"), F.lit("Other"))
).drop("category_key", "division_val")

# Show distinct categories and divisions
df_bronze.select("category", "division").distinct().show()

+-----------------+--------------------+
|         category|            division|
+-----------------+--------------------+
|   Healthy Snacks|      Healthy Snacks|
|      Energy Bars|      Nutrition Bars|
|Granola & Cereals|     Breakfast Foods|
|   Recovery Dairy|    Dairy & Recovery|
|     Protein Bars|      Nutrition Bars|
|  Electrolyte Mix|Hydration & Elect...|
+-----------------+--------------------+



In [0]:
# Extract text between brackets for variant
df_bronze = df_bronze.withColumn(
    "variant", 
    F.regexp_extract(F.col("product_name"), r"\((.*?)\)", 1)
)

# Show distinct product names and variants
df_bronze.show(truncate=False)

+-------------------------------------------------+----------+-----------------+-------------------------+------------+---------+------------------------+----------+
|product_name                                     |product_id|category         |read_timestamp           |file_name   |file_size|division                |variant   |
+-------------------------------------------------+----------+-----------------+-------------------------+------------+---------+------------------------+----------+
|SportsBar Oats Cookie Bites ChocoChip (180g)     |25891503  |Healthy Snacks   |2026-09-28 06:51:37.75369|products.csv|1388     |Healthy Snacks          |180g      |
|SportsBar Energy Bar Choco Fudge (40g)           |25891102  |Energy Bars      |2026-09-28 06:51:37.75369|products.csv|1388     |Nutrition Bars          |40g       |
|SportsBar Granola Crunch Honey Almond (300g)     |25891302  |Granola & Cereals|2026-09-28 06:51:37.75369|products.csv|1388     |Breakfast Foods         |300g      |
|Spo

In [0]:
df_bronze = (
    df_bronze
    # 1. Generate deterministic product_code from product_name using SHA-256
    .withColumn(
        "product_code",
        F.sha2(F.col("product_name").cast("string"), 256)
    )
    # 2. Clean product_id: keep numeric IDs, set non-numeric to default '999999'
    .withColumn(
        "product_id",
        F.when(
            F.col("product_id").cast("string").rlike("^[0-9]+$"),
            F.col("product_id").cast("string")
        ).otherwise(F.lit("999999"))
    )
    # 3. Rename product_name to product
    .withColumnRenamed("product_name", "product")
)

# Show Results
df_bronze.show()

+--------------------+----------+-----------------+--------------------+------------+---------+--------------------+----------+--------------------+
|             product|product_id|         category|      read_timestamp|   file_name|file_size|            division|   variant|        product_code|
+--------------------+----------+-----------------+--------------------+------------+---------+--------------------+----------+--------------------+
|SportsBar Oats Co...|  25891503|   Healthy Snacks|2026-09-28 06:51:...|products.csv|     1388|      Healthy Snacks|      180g|062f5574bbdf4386b...|
|SportsBar Energy ...|  25891102|      Energy Bars|2026-09-28 06:51:...|products.csv|     1388|      Nutrition Bars|       40g|e92c739a8d78cd6cb...|
|SportsBar Granola...|  25891302|Granola & Cereals|2026-09-28 06:51:...|products.csv|     1388|     Breakfast Foods|      300g|d9ebd1ca64d23951a...|
|SportsBar Greek Y...|  25891403|   Recovery Dairy|2026-09-28 06:51:...|products.csv|     1388|    Dairy &

## Silver Processing

In [0]:
df_bronze.write \
    .format('delta') \
    .option('delta.enableChangeDataFeed', 'true') \
    .mode('overwrite') \
    .saveAsTable(f'{catalog}.silver.sb_dim_{data_source}')

## Gold Processing

In [0]:
df_bronze.write \
    .format('delta') \
    .option('delta.enableChangeDataFeed', 'true') \
    .mode('overwrite') \
    .saveAsTable(f'{catalog}.gold.sb_dim_{data_source}')

In [0]:
# 1. Load staging products data and remove duplicate keys
child_prod_table = spark.sql(f'SELECT product_code, division, category, product, variant FROM {catalog}.gold.sb_dim_{data_source}')
child_deduped = child_prod_table.dropDuplicates(["product_code"])

# 2. Clean and deduplicate Target Parent Dimension Table
parent_table_name = f"{catalog}.gold.dim_products"
clean_parent_df = spark.read.table(parent_table_name).dropDuplicates(["product_code"])

clean_parent_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(parent_table_name)

# 3. Refresh DeltaTable instance
parent_prod_table = DeltaTable.forName(spark, parent_table_name)

# 4. Upsert (Merge) staging data into Dimension Table
parent_prod_table.alias("target").merge(
    source=child_deduped.alias("source"),
    condition="target.product_code = source.product_code"
).whenMatchedUpdate(
    set={
        "division": "source.division",
        "category": "source.category",
        "product": "source.product",
        "variant": "source.variant"
    }
).whenNotMatchedInsert(
    values={
        "product_code": "source.product_code",
        "division": "source.division",
        "category": "source.category",
        "product": "source.product",
        "variant": "source.variant"
    }
).execute()

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

In [0]:
# Display top records in dimension table
print('Top records in Parent Table :\n')
parent_prod_table.toDF().show(truncate=False)
print()

# Display hashed product records
print('Top records in Parent Table from Chilled Table :\n')
parent_prod_table.toDF().filter("length(product_code) = 64").show(truncate=False)

Top records in Parent Table :

+------------+-------------+--------------------+-----------------------------------+-----------------+
|product_code|division     |category            |product                            |variant          |
+------------+-------------+--------------------+-----------------------------------+-----------------+
|ARCH158F41  |Archery      |Finger Tab          |LX Leather Finger Tab              |Universal        |
|BASKED86BF  |Basketball   |Basketball Hoop     |HX Portable Basketball Hoop        |Adjustable Height|
|BOXB9DCC6   |Boxing       |Boxing Gloves       |BX Pro Boxing Gloves               |10 oz            |
|BOXB0228B   |Boxing       |Mouth Guard         |NX ShockGuard Mouthpiece           |Junior           |
|CAMP24F868  |Camping      |Tent                |CX QuickPitch Tent                 |4 Person         |
|CRIC2716B4  |Cricket      |Cricket Helmet      |AX Guard Cricket Helmet            |XL               |
|CRICB9EBFF  |Cricket      |Crick

In [0]:
# ==============================================================================
# Pipeline Quality Checks & Data Lineage Metrics (Bronze -> Silver -> Gold)
# ==============================================================================

print("=" * 70)
print(" 1. DATA LINEAGE & RECORD COUNTS (Bronze -> Silver -> Gold)")
print("=" * 70)

# Fetch record counts for each Medallion layer
bronze_cnt = spark.sql(f"SELECT COUNT(*) FROM {catalog}.bronze.sb_{data_source}").collect()[0][0]
silver_cnt = spark.sql(f"SELECT COUNT(*) FROM {catalog}.silver.sb_{data_source}").collect()[0][0]
gold_staging_cnt = spark.sql(f"SELECT COUNT(*) FROM {catalog}.gold.sb_{data_source}").collect()[0][0]
gold_dim_cnt = spark.sql(f"SELECT COUNT(*) FROM {catalog}.gold.dim_products").collect()[0][0]

print(f" Raw Bronze Records   : {bronze_cnt}")
print(f" Cleaned Silver Records: {silver_cnt} (Dropped duplicates: {bronze_cnt - silver_cnt})")
print(f" Gold Staging Records : {gold_staging_cnt}")
print(f" Main Dimension (Gold): {gold_dim_cnt} Total Unique Products")


print("\n" + "=" * 70)
print(" 2. DATA QUALITY & INTEGRITY CHECKS (Silver Layer)")
print("=" * 70)

# Check for Null values or fallback IDs in Silver Layer
df_silver_check = spark.sql(f"SELECT * FROM {catalog}.silver.sb_{data_source}")

null_categories = df_silver_check.filter(F.col("category").isNull()).count()
null_divisions = df_silver_check.filter(F.col("division").isNull() | (F.col("division") == "Other")).count()
fallback_product_ids = df_silver_check.filter(F.col("product_id") == "999999").count()

print(f" Null Categories Count     : {null_categories} {'[PASSED]' if null_categories == 0 else '[WARNING]'}")
print(f" Unmapped/Other Divisions  : {null_divisions} {'[PASSED]' if null_divisions == 0 else '[WARNING]'}")
print(f" Fallback Product IDs (999999): {fallback_product_ids} [ACCEPTED BY DESIGN]")


print("\n" + "=" * 70)
print(" 3. PRIMARY KEY UNIQUE & FORMAT VALIDATION (Gold Dimension)")
print("=" * 70)

# Check Primary Key Uniqueness and Code Format Mix
df_gold_dim = spark.table(f"{catalog}.gold.dim_products")

duplicate_keys = df_gold_dim.groupBy("product_code").count().filter(F.col("count") > 1).count()
legacy_hashes_cnt = df_gold_dim.filter(F.length(F.col("product_code")) != 64).count()
sha256_hashes_cnt = df_gold_dim.filter(F.length(F.col("product_code")) == 64).count()

print(f" Duplicate Product Codes   : {duplicate_keys} {'[PASSED]' if duplicate_keys == 0 else '[FAILED]'}")
print(f" Product Code Format Split : {sha256_hashes_cnt} SHA-256 / {legacy_hashes_cnt} Legacy [ACCEPTED]")


print("\n" + "=" * 70)
print(" 4. SAMPLE SUMMARY OF FINAL TRANSFORMED GOLD DATA")
print("=" * 70)

# Display Division Breakdown from Gold Dimension Table
df_gold_dim.groupBy("division", "category") \
    .agg(F.count("product_code").alias("total_products")) \
    .orderBy("division", "category") \
    .show(truncate=False)

 1. DATA LINEAGE & RECORD COUNTS (Bronze -> Silver -> Gold)
 Raw Bronze Records   : 20
 Cleaned Silver Records: 18 (Dropped duplicates: 2)
 Gold Staging Records : 18
 Main Dimension (Gold): 415 Total Unique Products

 2. DATA QUALITY & INTEGRITY CHECKS (Silver Layer)
 Null Categories Count     : 0 [PASSED]
 Unmapped/Other Divisions  : 0 [PASSED]
 Fallback Product IDs (999999): 1 [ACCEPTED BY DESIGN]

 3. PRIMARY KEY UNIQUE & FORMAT VALIDATION (Gold Dimension)
 Duplicate Product Codes   : 0 [PASSED]
 Product Code Format Split : 18 SHA-256 / 397 Legacy [ACCEPTED]

 4. SAMPLE SUMMARY OF FINAL TRANSFORMED GOLD DATA
+---------------+----------------------+--------------+
|division       |category              |total_products|
+---------------+----------------------+--------------+
|Archery        |Arm Guard             |2             |
|Archery        |Arrows                |1             |
|Archery        |Bow                   |2             |
|Archery        |Bow Stringer          |1    